# Extra research

https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html?

Thresholding in simple terms



In [4]:
#imports 
import cv2
import numpy as np
import tensorflow as tf
from collections import Counter

MODEL_PATH = "rice_classifier.keras"
IMAGE_PATH = "handful.jpg"
IMG_SIZE = 224
MIN_AREA = 100
CLASS_NAMES = ["basmati", "jasmine", "brown", "sushi"]

In [11]:
# load my model in 
model = tf.keras.models.load_model(MODEL_PATH)
print(model)
model.summary()

<Sequential name=sequential, built=True>


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (64, 224, 224, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (64, 224, 224, 16)     │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (64, 112, 112, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (64, 112, 112, 32)     │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (64, 56, 56, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (64, 56, 56, 64)       │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (64, 28, 28, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (64, 50176)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 128)              │     6,422,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (64, 5)                │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,464,631 (24.66 MB)

 Trainable params: 6,464,629 (24.66 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [ ]:
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

original = image.copy()

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

_, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

white_ratio = np.mean(mask == 255)
if white_ratio > 0.7:
    mask = cv2.bitwise_not(mask)

kernel = np.ones((3, 3), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

In [ ]:
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

results = []
counts = Counter()

In [ ]:
for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    if area < MIN_AREA:
        continue

    x, y, w, h = cv2.boundingRect(contour)

    crop = original[y:y+h, x:x+w]
    crop_mask = mask[y:y+h, x:x+w]

    masked_crop = cv2.bitwise_and(crop, crop, mask=crop_mask)
    grain_gray = cv2.cvtColor(masked_crop, cv2.COLOR_BGR2GRAY)
    grain_resized = cv2.resize(grain_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    grain_resized = grain_resized.astype(np.float32) / 255.0
    x_input = np.expand_dims(grain_resized, axis=(0, -1))
    # run our predition 
    probs = model.predict(x_input, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])

    counts[pred_label] += 1
    results.append({
        "box": (x, y, w, h),
        "label": pred_label,
        "confidence": confidence
    })

    cv2.rectangle(original, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(
        original,
        f"{pred_label} {confidence:.2f}",
        (x, max(20, y - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
        cv2.LINE_AA
    )

In [ ]:
print("Detections:")
for r in results:
    print(r)

print("\nCounts:")
print(dict(counts))

cv2.imwrite("mask.png", mask)
cv2.imwrite("annotated.png", original)

print("\nSaved:")
print("- mask.png")
print("- annotated.png")